In [ ]:
#|default_exp test_cli_graph

In [ ]:
#|hide
from nblite import nbl_export; nbl_export();

In [ ]:
#|export
import json
from pathlib import Path

import pytest
from typer.testing import CliRunner

from netrun_cli._app import app

In [ ]:
#|export
runner = CliRunner()


def _make_config(tmp_path: Path, data: dict) -> str:
    """Write a temp .netrun.json and return its path as a string."""
    p = tmp_path / "test.netrun.json"
    p.write_text(json.dumps(data, indent=2))
    return str(p)


def _base_config() -> dict:
    """Minimal config with two nodes and one edge."""
    return {
        "graph": {
            "nodes": [
                {
                    "name": "A",
                    "in_ports": {"in": {}},
                    "out_ports": {"out": {}},
                    "extra": {"ui": {"position": {"x": 100, "y": 200}}},
                },
                {
                    "name": "B",
                    "in_ports": {"in": {}},
                    "out_ports": {"out": {}},
                    "extra": {"ui": {"position": {"x": 400, "y": 200}}},
                },
            ],
            "edges": [
                {"source_node": "A", "source_port": "out", "target_node": "B", "target_port": "in"},
            ],
        }
    }

## Test add-node

In [ ]:
#|export
def test_add_node_factory(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, [
        "add-node", "C", "-c", cfg,
        "--factory", "netrun.node_factories.from_function",
        "--factory-arg", "func=my_module.my_func",
        "--no-validate",
    ])
    assert result.exit_code == 0, result.output
    node = json.loads(result.stdout)
    assert node["name"] == "C"
    assert node["factory"] == "netrun.node_factories.from_function"
    assert node["factory_args"]["func"] == "my_module.my_func"
    # Check it was written
    written = json.loads(Path(cfg).read_text())
    names = [n["name"] for n in written["graph"]["nodes"]]
    assert "C" in names


def test_add_node_raw_ports(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, [
        "add-node", "C", "-c", cfg,
        "--in-ports", "x,y",
        "--out-ports", "result",
        "--no-validate",
    ])
    assert result.exit_code == 0, result.output
    node = json.loads(result.stdout)
    assert "x" in node["in_ports"]
    assert "y" in node["in_ports"]
    assert "result" in node["out_ports"]


def test_add_node_json_stdin(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    node_json = json.dumps({"in_ports": {"data": {}}, "out_ports": {"result": {}}, "factory": "some.factory"})
    result = runner.invoke(app, [
        "add-node", "C", "-c", cfg, "--json", "--no-validate",
    ], input=node_json)
    assert result.exit_code == 0, result.output
    node = json.loads(result.stdout)
    assert node["name"] == "C"
    assert node["factory"] == "some.factory"
    assert "data" in node["in_ports"]


def test_add_node_duplicate_error(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["add-node", "A", "-c", cfg, "--no-validate"])
    assert result.exit_code == 1
    assert "already exists" in result.output


def test_add_node_custom_position(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, [
        "add-node", "C", "-c", cfg,
        "--position", "500,300",
        "--no-validate",
    ])
    assert result.exit_code == 0, result.output
    node = json.loads(result.stdout)
    assert node["extra"]["ui"]["position"]["x"] == 500.0
    assert node["extra"]["ui"]["position"]["y"] == 300.0


def test_add_node_auto_position(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["add-node", "C", "-c", cfg, "--no-validate"])
    assert result.exit_code == 0, result.output
    node = json.loads(result.stdout)
    # max x is 400, so auto should be 700; avg y is 200
    assert node["extra"]["ui"]["position"]["x"] == 700.0
    assert node["extra"]["ui"]["position"]["y"] == 200.0

## Test remove-node

In [ ]:
#|export
def test_remove_node_with_edges(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["remove-node", "A", "-c", cfg, "--no-validate"])
    assert result.exit_code == 0, result.output
    out = json.loads(result.stdout)
    assert out["removed"] == "A"
    assert out["edges_removed"] == 1
    # Verify config
    written = json.loads(Path(cfg).read_text())
    names = [n["name"] for n in written["graph"]["nodes"]]
    assert "A" not in names
    assert len(written["graph"]["edges"]) == 0


def test_remove_node_not_found(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["remove-node", "Z", "-c", cfg, "--no-validate"])
    assert result.exit_code == 1
    assert "not found" in result.output

## Test edit-node

In [ ]:
#|export
def test_edit_node_rename(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["edit-node", "A", "-c", cfg, "--rename", "Alpha", "--no-validate"])
    assert result.exit_code == 0, result.output
    node = json.loads(result.stdout)
    assert node["name"] == "Alpha"
    # Verify edge refs updated
    written = json.loads(Path(cfg).read_text())
    edge = written["graph"]["edges"][0]
    assert edge["source_node"] == "Alpha"


def test_edit_node_ports(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, [
        "edit-node", "A", "-c", cfg,
        "--add-in-port", "extra_in",
        "--add-out-port", "extra_out",
        "--remove-in-port", "in",
        "--no-validate",
    ])
    assert result.exit_code == 0, result.output
    node = json.loads(result.stdout)
    assert "extra_in" in node["in_ports"]
    assert "extra_out" in node["out_ports"]
    assert "in" not in node["in_ports"]


def test_edit_node_merge(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    merge_json = json.dumps({"execution_config": {"pools": ["main"]}})
    result = runner.invoke(app, [
        "edit-node", "A", "-c", cfg,
        "--merge", merge_json,
        "--no-validate",
    ])
    assert result.exit_code == 0, result.output
    node = json.loads(result.stdout)
    assert node["execution_config"]["pools"] == ["main"]
    # Original fields preserved
    assert node["name"] == "A"
    assert "out" in node["out_ports"]


def test_edit_node_not_found(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["edit-node", "Z", "-c", cfg, "--rename", "ZZ", "--no-validate"])
    assert result.exit_code == 1
    assert "not found" in result.output

## Test add-edge

In [ ]:
#|export
def test_add_edge_basic(tmp_path):
    data = _base_config()
    data["graph"]["edges"] = []  # Start with no edges
    cfg = _make_config(tmp_path, data)
    result = runner.invoke(app, ["add-edge", "A", "out", "B", "in", "-c", cfg, "--no-validate"])
    assert result.exit_code == 0, result.output
    edge = json.loads(result.stdout)
    assert edge["source_node"] == "A"
    assert edge["source_port"] == "out"
    assert edge["target_node"] == "B"
    assert edge["target_port"] == "in"
    # Verify written
    written = json.loads(Path(cfg).read_text())
    assert len(written["graph"]["edges"]) == 1


def test_add_edge_dependency(tmp_path):
    data = _base_config()
    data["graph"]["edges"] = []
    cfg = _make_config(tmp_path, data)
    result = runner.invoke(app, ["add-edge", "A", "out", "B", "in", "-c", cfg, "--dependency", "--no-validate"])
    assert result.exit_code == 0, result.output
    edge = json.loads(result.stdout)
    assert edge["dependency"] is True


def test_add_edge_fan_out_warning(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    # A.out already has an edge; adding another should warn
    # First add a third node
    runner.invoke(app, ["add-node", "C", "-c", cfg, "--in-ports", "in", "--no-validate"])
    result = runner.invoke(app, ["add-edge", "A", "out", "C", "in", "-c", cfg, "--no-validate"])
    assert result.exit_code == 0, result.output
    assert "fan-out" in result.output.lower() or "Fan-out" in result.output


def test_add_edge_missing_node(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["add-edge", "Z", "out", "B", "in", "-c", cfg, "--no-validate"])
    assert result.exit_code == 1
    assert "not found" in result.output

## Test remove-edge

In [ ]:
#|export
def test_remove_edge_basic(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["remove-edge", "A", "out", "B", "in", "-c", cfg, "--no-validate"])
    assert result.exit_code == 0, result.output
    out = json.loads(result.stdout)
    assert out["removed_source_node"] == "A"
    assert out["removed_source_port"] == "out"
    # Verify
    written = json.loads(Path(cfg).read_text())
    assert len(written["graph"]["edges"]) == 0


def test_remove_edge_not_found(tmp_path):
    cfg = _make_config(tmp_path, _base_config())
    result = runner.invoke(app, ["remove-edge", "X", "out", "Y", "in", "-c", cfg, "--no-validate"])
    assert result.exit_code == 1
    assert "not found" in result.output

## Test fan-out validation message

In [ ]:
#|export
def test_validate_fan_out_suggestion():
    from netrun.net.config._graph import GraphConfig
    from netrun.net.config._nodes import NodeConfig, PortConfig, EdgeConfig

    config = GraphConfig(
        nodes=[
            NodeConfig(name="A", out_ports={"out": PortConfig()}),
            NodeConfig(name="B", in_ports={"in": PortConfig()}),
            NodeConfig(name="C", in_ports={"in": PortConfig()}),
        ],
        edges=[
            EdgeConfig(source_node="A", source_port="out", target_node="B", target_port="in"),
            EdgeConfig(source_node="A", source_port="out", target_node="C", target_port="in"),
        ],
    )
    errors = config.validate()
    fan_out_errors = [e for e in errors if e.type == "fan_out"]
    assert len(fan_out_errors) > 0
    assert "netrun.node_factories.broadcast" in fan_out_errors[0].msg